# AI Kids Story Studio - character reference review (v3)

    **Not GPU-executed during implementation.** The orchestration/state/import code is locally tested with synthetic fixtures.
    This launcher uses the SAME worker code shipped by your local production; no separate generation loop lives in the notebook.
    It cannot start a GPU runtime for you. Start on a CPU runtime for dry-run validation, then explicitly choose a GPU for generation.
    Upload the production `image_handoff/request_*.zip` ONCE. It contains the exact request, available references and worker source.
    Outputs, checkpoints, reference copies and exports live in Drive. Reopen the same package/result directory to resume.
    Default is `dry_run`: no model imports/downloads/inference and no result mutation.

    Use the NEW `examples/character_references_v3.zip`. Both v2 references FAILED human review; retain them unchanged.
    Generate ONLY two NEW single-view reference candidates with `MODE="references"`, budget 2. There are ZERO scene jobs.
    Challenge, smoke and full generation are disabled by this request. Do not approve candidates automatically.
    References use text-only SDXL, not IP-Adapter. They share a studio-render style; color/count/style remain subject to human review.
    Fresh-reference policy excludes all external candidates. Successful outputs of this exact batch resume without regeneration.
    For an individual rejection, use a new single-character/attempt handoff (see examples/README.md), not the failed result directory.


Checkpoints and compatibility checked against their published model cards/docs:
- [SDXL Base 1.0](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0):
  [CreativeML Open RAIL++-M](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/raw/main/LICENSE.md).
- [SD 1.5 fallback](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5):
  CreativeML OpenRAIL-M (use restrictions apply; this is a community mirror, not affiliated with Runway).
- [h94/IP-Adapter](https://huggingface.co/h94/IP-Adapter): Apache-2.0.
  SDXL uses `sdxl_models/ip-adapter_sdxl_vit-h.safetensors` plus `models/image_encoder` (ViT-H).
  SD 1.5 uses `models/ip-adapter_sd15.safetensors` with its matching ViT-H encoder.
- [Diffusers 0.35.1 adapter documentation](https://huggingface.co/docs/diffusers/v0.35.1/en/using-diffusers/ip_adapter),
  [memory guidance](https://huggingface.co/docs/diffusers/v0.35.1/en/optimization/memory).

These are open weights with license conditions, not public-domain checkpoints. Read the linked terms before use/redistribution.
IP-Adapter guides appearance; it does **not guarantee** animal identity, clothing, shell patterns or two-character separation.
This revision encodes references separately with regional attention masks; independent identity is still not guaranteed.
Review every result; no output is approved automatically.

In [ ]:
# 1. Controls. Rerun this cell only after deliberately editing these values.
MODE = "dry_run"  # Only dry_run or references for the v3 reference-only package.
MAX_NEW_IMAGES = 2  # per invocation, successful existing images do not count
SMOKE_SCENE_ID = "challenge"  # unused: the v3 request has no scenes
RUN_GENERATION = False
INSTALL_COLAB_DEPS = False
LICENSES_REVIEWED = False
ALLOW_UNAPPROVED_REFERENCE_PREVIEW = False
QUALITY_REVIEWED = False  # Human smoke-quality acknowledgement, not production asset approval
REFERENCES_REVIEWED = False  # No scene generation at this milestone
FAMILY = "sdxl"  # sd15 is the smaller explicit fallback; never silently substituted
STEPS = 30
BASE_SEED = 20261124  # v3 attempt 1; attempt 2 requires 20261125 (checked against request)
IP_ADAPTER_SCALE = 0.35  # NOT applied to text-only reference generation
PACKAGE_PATH = ""  # Set to the saved Drive ZIP path on subsequent sessions; otherwise upload once
RESULT_BATCH_ID = ""  # blank => NEW request/settings/implementation identity; NEVER use a v2 result directory
REFERENCE_BATCH_MANIFEST = ""  # keep blank; v3 fresh-reference requests exclude external candidates
REFERENCE_IDS = {}  # only needed for ambiguous refs: {"hare": "exact candidate ID or sha256"}

In [ ]:
# 2. Mount Drive, validate/extract the small handoff, load shared worker (NO MODELS).
import json, hashlib, sys, uuid, zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive, files
drive.mount("/content/drive")
SHARED = Path("/content/drive/MyDrive/AI_Kids_Story_Studio/batch_handoff")
SHARED.mkdir(parents=True, exist_ok=True)
if not PACKAGE_PATH:
    uploaded = files.upload()  # Choose character_references_v3.zip (or a new individual-reference retry ZIP).
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one handoff ZIP")
    payload = next(iter(uploaded.values()))
    package_hash = hashlib.sha256(payload).hexdigest()
    package = SHARED / (package_hash + ".zip")
    if not package.exists():
        with package.open("xb") as stream:
            stream.write(payload)
else:
    package = Path(PACKAGE_PATH)
    package_hash = hashlib.sha256(package.read_bytes()).hexdigest()
bundle = SHARED / package_hash
with zipfile.ZipFile(package) as archive:
    if sum(m.file_size for m in archive.infolist()) > 128 * 1024 * 1024 or len(archive.infolist()) > 200:
        raise ValueError("Unexpectedly large handoff")
    index = json.loads(archive.read("bundle_index.json"))
    expected = set(index["files"]) | {"bundle_index.json"}
    if set(archive.namelist()) != expected:
        raise ValueError("Unexpected handoff files")
    for name in archive.namelist():
        relative = PurePosixPath(name)
        if relative.is_absolute() or ".." in relative.parts or "\\" in name or ":" in name:
            raise ValueError("Unsafe ZIP path")
        payload = archive.read(name)
        if name != "bundle_index.json" and hashlib.sha256(payload).hexdigest() != index["files"][name]:
            raise ValueError("Handoff checksum mismatch")
        target = bundle / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            if target.read_bytes() != payload:
                raise ValueError("Existing persistent bundle was modified; refusing to overwrite")
        else:
            with target.open("xb") as stream:
                stream.write(payload)
previous_module = sys.modules.get("src.image_batch")
if previous_module and not Path(previous_module.__file__).resolve().is_relative_to(bundle.resolve()):
    raise RuntimeError("A different bundle is already imported. Restart the session before loading new code.")
sys.path.insert(0, str(bundle))
request_data = json.loads((bundle / "request.json").read_text())
if (request_data.get("reference_prompt_revision") != "single_reference_v3" or
    request_data.get("generation_policy", {}).get("reference_only") is not True or request_data.get("scenes") != []):
    raise RuntimeError("Upload the NEW character_references_v3.zip or a v3 single-reference retry; scenes are disabled.")
from src.image_batch import BatchWorker, BatchSettings, reference_sources, contact_sheet, generation_identity
request_hash = hashlib.sha256((bundle / "request.json").read_bytes()).hexdigest()
if request_hash != index["request_sha256"]:
    raise ValueError("Production request hash mismatch")
settings = BatchSettings(family=FAMILY, steps=STEPS, base_seed=BASE_SEED, adapter_scale=IP_ADAPTER_SCALE,
                         reference_size=request_data["generation_config"]["reference_size"])
generation_key = generation_identity(request_hash, settings)
result_name = RESULT_BATCH_ID or ("references_v3_" + generation_key[:16])
if not result_name.replace("_", "").replace("-", "").isalnum():
    raise ValueError("Use a simple alphanumeric result batch name")
result_dir = SHARED / "results" / result_name
print("Saved PACKAGE_PATH for resume:", package)
print("Persistent result directory:", result_dir)

In [ ]:
# 3. Preflight before installing/loading any GPU model. Read-only dry runs are safe.
settings = BatchSettings(family=FAMILY, steps=STEPS, base_seed=BASE_SEED, adapter_scale=IP_ADAPTER_SCALE,
                         reference_size=request_data["generation_config"]["reference_size"])
references = reference_sources(bundle / "references.json")
if REFERENCE_BATCH_MANIFEST:
    references += reference_sources(Path(REFERENCE_BATCH_MANIFEST))
worker = BatchWorker(bundle / "request.json", result_dir, references=references, settings=settings)
controls = dict(mode=MODE, smoke_scene=SMOKE_SCENE_ID, max_new_images=MAX_NEW_IMAGES,
    allow_unapproved_references=ALLOW_UNAPPROVED_REFERENCE_PREVIEW,
    quality_reviewed=QUALITY_REVIEWED, reference_ids=REFERENCE_IDS)
controls["references_reviewed"] = REFERENCES_REVIEWED
preflight = worker.preflight(**controls)
print(json.dumps({k: v for k, v in preflight.items() if k != "chosen_references"}, indent=2))
print("Review count, reused images, warnings and blockers ABOVE before enabling GPU execution.")

In [ ]:
# 4. Explicit execution: references ONLY. Scene generation is blocked by the request.
if MODE == "dry_run":
    outcome = worker.execute(lambda *args: (_ for _ in ()).throw(RuntimeError("Dry-run must not load a model")), **controls)
    print("DRY RUN ONLY:", json.dumps(outcome, indent=2))
else:
    if preflight["blockers"]:
        raise RuntimeError("Resolve preflight blockers first: " + "; ".join(preflight["blockers"]))
    if not RUN_GENERATION or not LICENSES_REVIEWED:
        raise RuntimeError("Explicitly set RUN_GENERATION and LICENSES_REVIEWED after inspecting preflight/license terms.")
    if preflight["new_image_count"]:
        if not INSTALL_COLAB_DEPS:
            raise RuntimeError("Enable INSTALL_COLAB_DEPS for this Colab-only inference run.")
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "diffusers==0.35.1",
            "transformers==4.56.2", "accelerate==1.10.1", "safetensors==0.6.2",
            "huggingface-hub==0.34.4", "Pillow>=10.4,<13"], check=True)
    from src.colab_images import ColabDiffusersBackend
    # Reuse one loaded pipeline for an interrupted reference batch in this session.
    # The persistent manifest still enables fresh-session resume after disconnect.
    if "_COLAB_BACKEND" not in globals():
        _COLAB_BACKEND = None
    def load_backend(settings, previous):
        global _COLAB_BACKEND
        if _COLAB_BACKEND is not None:
            if _COLAB_BACKEND.settings != settings:
                raise RuntimeError("Changed model settings: restart the runtime and choose a new result batch.")
            if previous and previous.get("model_revision") != _COLAB_BACKEND.provenance["model_revision"]:
                raise RuntimeError("Loaded model revision differs from the persistent batch; restart the session.")
            return _COLAB_BACKEND
        _COLAB_BACKEND = ColabDiffusersBackend(settings, previous)
        return _COLAB_BACKEND
    outcome = worker.execute(load_backend, **controls)
    print(json.dumps({k: outcome.get(k) for k in ("status", "completion", "errors")}, indent=2))
    if outcome["status"] == "failed":
        raise RuntimeError("Batch failed. Read its saved errors; successful images remain reusable. Export partial results if needed.")

In [ ]:
# 5. One labeled review sheet for the requested reference(s), including missing/failed panels.
from IPython.display import display
from PIL import Image
if (result_dir / "manifest.json").exists():
    review_path = contact_sheet(result_dir)
    display(Image.open(review_path))
    print("Review:", review_path)
else:
    print("Dry run has no result images; no contact sheet generated.")

In `MODE="references"`, verify ONE full-body three-quarter white rabbit: pink inner ears, small pink nose,
BLUE sleeveless vest, no bow tie or other clothing. Verify ONE entire green tortoise in three-quarter side view:
large rounded BROWN back shell with defined hexagonal scutes. Both must share the friendly 3D studio-render style.
No repeated figures, tiles, wallpaper, panels, character sheets, cropped bodies, text or watermark.
`complete` means files generated and checksums verified, NEVER visual approval. Both remain candidates.
If either fails, STOP and retain all files. Use the builder's `--characters tortoise --attempt 2` (or hare)
in a NEW handoff, update BASE_SEED to its printed seed and use a fresh session with blank RESULT_BATCH_ID.
Only the rejected reference is scheduled; the good candidate remains unchanged in its original batch.
Do NOT generate challenge yet, even if both look good. Export with cell 6 and stop for human review.

A CUDA/adapter failure stops the invocation and is recorded without a text-only fallback or automatic costly retry.
For OOM, export partial results; use a new batch with `FAMILY="sd15"` in a fresh runtime. That explicit fallback
renders 512-square references and may have lower quality. It does not load an adapter for references.
No ControlNet, refiner, fine-tuning or animation model is loaded. The shared worker still supports the existing
scene-conditioning architecture, but this reference-only request cannot run it.

In [ ]:
# 6. Export once per invocation; filenames are unique, all originals remain on Drive.
import shutil
if not (result_dir / "manifest.json").exists():
    print("Nothing generated; dry-run does not export a fake result.")
else:
    export_root = SHARED / "exports"
    export_root.mkdir(exist_ok=True)
    archive_base = export_root / (result_dir.name + "_" + uuid.uuid4().hex[:12])
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=result_dir)
    print("Persistent export:", archive_path)
    files.download(archive_path)
    print("Disconnect/delete the GPU runtime now when finished, to stop idle credit use.")